In [ ]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from scipy.linalg import eigh

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import fastplotlib as fpl

import optuna

from dysts.maps import Henon

# Init

In [14]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [15]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [16]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [17]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.shape[1] >= 3:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [18]:
scatter_plot([henon_dataset], ["Henon"], ["magenta"])

In [8]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [9]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [10]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [168]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=15,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

# Calc Init

In [817]:
# @njit(fastmath=True, cache=True)
def create_stiffness_matrix(node_positions, connections, k_vals):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for node_conn, k_val in zip(connections, k_vals):
        node_pos = node_positions[node_conn]
        diff_vec = node_pos[1] - node_pos[0]
        unit_dir = diff_vec / np.linalg.norm(diff_vec)
        sub_block = np.outer(unit_dir, unit_dir)

        idx1 = node_conn[0] * dims
        idx2 = node_conn[1] * dims

        K[idx1 : idx1 + dims, idx1 : idx1 + dims] += k_val * sub_block
        K[idx2 : idx2 + dims, idx2 : idx2 + dims] += k_val * sub_block
        K[idx1 : idx1 + dims, idx2 : idx2 + dims] += k_val * -sub_block
        K[idx2 : idx2 + dims, idx1 : idx1 + dims] += k_val * -sub_block
    return K

In [663]:
@njit(fastmath=True, cache=True)
def run_simulation(
    steps,
    dt,
    matrix_size,
    M_INV,
    C,
    U,
    initial_pos,
    connections_list,
    k_vals,
    constrained_nodes=[-1],
    wall_nodes=[-1],
    constrained_values=0,
):
    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    dims = initial_pos.shape[1]

    for i in range(1, steps):
        actual_pos = initial_pos + disp[i - 1].reshape(-1, dims)
        K = create_stiffness_matrix(actual_pos, connections_list, k_vals)
        if constrained_nodes[0] != -1:
            for node in constrained_nodes:
                idx = node * dims
                for j in range(dims):
                    K[idx + j, idx + j] += constrained_values
        if wall_nodes[0] != -1:
            for node in wall_nodes:
                idx = node * dims
                for j in range(dims):
                    K[idx + j, :] = 0
                    K[:, idx + j] = 0

        acc = M_INV @ (-K @ disp[i - 1] - C @ v[i - 1] + U[i - 1])

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ disp[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

In [830]:
# @njit(fastmath=True, cache=True)
def get_spring_forces(connections_list, disp, initial_pos, k_vals, matrix_size, dims):
    forces = np.zeros(matrix_size)
    for i in range(len(connections_list)):
        idx_a, idx_b = connections_list[i]
        start_a, end_a = idx_a * dims, (idx_a + 1) * dims
        start_b, end_b = idx_b * dims, (idx_b + 1) * dims
        pos_a = initial_pos[idx_a] + disp[start_a:end_a]
        pos_b = initial_pos[idx_b] + disp[start_b:end_b]
        r_vec = pos_b - pos_a
        current_len = np.linalg.norm(r_vec)
        if current_len < 1e-13:
            continue
        init_vec = initial_pos[idx_b] - initial_pos[idx_a]
        rest_len = np.linalg.norm(init_vec)
        force_magnitude = k_vals[i] * (current_len - rest_len)
        unit_dir = r_vec / current_len
        spring_force_vector = force_magnitude * unit_dir
        forces[start_a:end_a] += spring_force_vector
        forces[start_b:end_b] -= spring_force_vector

    return forces

In [831]:
# @njit(fastmath=True, cache=True)
def run_simulation_2(
    steps,
    dt,
    M_INV,
    C,
    U,
    initial_pos,
    connections_list,
    k_vals,
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        F_spring_last = get_spring_forces(
            connections_list, disp[i - 1], initial_pos, k_vals, matrix_size, dims
        )

        acc = M_INV @ (F_spring_last - C @ v[i - 1] + U[i - 1])

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, k_vals, matrix_size, dims
        )

        acc_next = M_INV @ (F_spring - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

In [ ]:
displacement, velocity = run_simulation_2(
    steps + transient_steps_reservoir + tau_steps,
    0.001,
    M_INV,
    DAMP,
    U,
    nodes_pos,
    connections_list,
    k_vals,
)

X = np.column_stack((displacement, velocity))

: 

# Fix Spring Math

In [797]:
N = 3

x = np.arange(N) 
nodes_pos = x.reshape(-1, 1)

node_ids = np.arange(N)
connections_list = np.column_stack((node_ids[:-1], node_ids[1:]))

In [798]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = np.ones(num_nodes) * 0.1
m_diag = np.repeat(node_m, dims)
m_inv_diag = 1.0 / m_diag
M = np.diag(m_diag)
M_INV = np.diag(m_inv_diag)

node_c = np.ones(num_nodes) * 0.1
c_diag = np.repeat(node_c, dims)
DAMP = np.diag(c_diag)

k_vals = np.ones(N - 1)

In [799]:
force = 100
U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = np.array([2])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
vectorized_force[0, 0] = force
U[:, col_indices] = vectorized_force

In [811]:
displacement, velocity = run_simulation_2(
    steps + transient_steps_reservoir + tau_steps,
    0.001,
    M_INV,
    DAMP,
    U,
    nodes_pos,
    connections_list,
    k_vals,
)

X = np.column_stack((displacement, velocity))

In [812]:
X[1]

array([0.000000e+00, 0.000000e+00, 5.000000e-04, 0.000000e+00,
       2.500000e-06, 4.994975e-01])

In [813]:
X[-1]

array([ 1.66667359e-01,  1.66666372e-01,  1.66666269e-01, -3.02228218e-06,
       -2.12532639e-06,  5.14774804e-06])

In [803]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps,
    0.001,
    matrix_size,
    M_INV,
    DAMP,
    U,
    nodes_pos,
    connections_list,
    k_vals,
    constrained_nodes=[-1],
    wall_nodes=[-1],
    constrained_values=0,
)

X = np.column_stack((displacement, velocity))

In [804]:
X[1]

array([0.000000e+00, 0.000000e+00, 5.000000e-04, 0.000000e+00,
       2.500000e-06, 4.994975e-01])

In [805]:
X[-1]

array([ 1.66667359e-01,  1.66666372e-01,  1.66666269e-01, -3.02228218e-06,
       -2.12532639e-06,  5.14774804e-06])

In [796]:
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
    vel=velocity,
    steps_jump=3
).show()

# Single Hex

In [819]:
side_len = 1
x = np.array(
    [-side_len / 2, side_len / 2, side_len, side_len / 2, -side_len / 2, -side_len]
)
y = np.array(
    [
        0,
        0,
        side_len * np.sqrt(3) / 2,
        side_len * np.sqrt(3),
        side_len * np.sqrt(3),
        side_len * np.sqrt(3) / 2,
    ]
)
nodes_pos = np.column_stack((x, y))

plot_grid(nodes_pos).show()

In [820]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m= rng.uniform(0.1, 0.3, size=num_nodes)
m_diag = np.repeat(node_m, dims)
m_inv_diag = 1.0 / m_diag
M = np.diag(m_diag)
M_INV = np.diag(m_inv_diag)

node_c = rng.uniform(0.05, 0.3, size=num_nodes)
c_diag = np.repeat(node_c, dims)
DAMP = np.diag(c_diag)

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = np.array([2, 5])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
vectorized_force[:, 0] = henon_scaled[:, 0]
vectorized_force[:, 2] = henon_scaled[:, 1]
U[:, col_indices] = vectorized_force

In [821]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, 1)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [ ]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps,
    0.01,
    matrix_size,
    M_INV,
    DAMP,
    U * 3,
    nodes_pos,
    connections_list,
    k_vals,
    [0, 1, 3, 4],
    [-1],
    100
)

X = np.column_stack((displacement, velocity))

In [ ]:
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    henon_train_scaled[transient_steps_reservoir + tau_steps :],
    henon_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.0292 0.4653


In [ ]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
).show()

## Optuna

In [ ]:
def hyper_param_input(input_force):
    displacement, velocity = run_simulation(
        steps + transient_steps_reservoir + tau_steps,
        0.01,
        matrix_size,
        M_INV,
        DAMP,
        U * input_force,
        nodes_pos,
        connections_list,
        k_vals,
        [0, 1, 3, 4],
        100,
    )
    X = np.column_stack((displacement, velocity))

    X_delayed = X[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    x_scaler = StandardScaler()
    X_train_scaled, X_test = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )
    Y_train_scaled, Y_test_scaled = (
        henon_train_scaled[transient_steps_reservoir + tau_steps :],
        henon_test_scaled,
    )
    model = RidgeCV()
    model.fit(X_train_scaled, Y_train_scaled)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_scaled)

    return Y_test, Y_pred

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    input_force = trial.suggest_float("input_force", 0.1, 100.0)

    Y_test, Y_pred = hyper_param_input(input_force)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[I 2026-07-08 09:50:29,749] A new study created in memory with name: no-name-03d6de9d-3b49-4ef0-925e-675046d89682


[Optuna] Processing Trial #99...

In [ ]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #92
  Values: [0.1605379320101885, 0.44478383859565773]
  Params: {'input_force': 0.10924010548283275}


In [ ]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps,
    0.01,
    matrix_size,
    M_INV,
    DAMP,
    U * study.best_trials[0].params["input_force"],
    nodes_pos,
    connections_list,
    k_vals,
    [0, 1, 3, 4],
    100,
)

Y_test, Y_pred = hyper_param_input(study.best_trials[0].params["input_force"])
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
r_2, mse

(0.1605379320101885, 0.44478383859565773)

In [ ]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
).show()

## Optuna

In [ ]:
def hyper_param_input(input_force):
    displacement, velocity = run_simulation(
        steps + transient_steps_reservoir + tau_steps,
        0.01,
        matrix_size,
        M_INV,
        DAMP,
        U * input_force,
        nodes_pos,
        connections_list,
        k_vals,
        [0, 1, 3, 4],
        100,
    )
    X = np.column_stack((displacement, velocity))

    X_delayed = X[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    x_scaler = StandardScaler()
    X_train_scaled, X_test = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )
    Y_train_scaled, Y_test_scaled = (
        henon_train_scaled[transient_steps_reservoir + tau_steps :],
        henon_test_scaled,
    )
    model = RidgeCV()
    model.fit(X_train_scaled, Y_train_scaled)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_scaled)

    return Y_test, Y_pred

In [ ]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    input_force = trial.suggest_float("input_force", 2, 30.0)

    Y_test, Y_pred = hyper_param_input(input_force)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [ ]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #73
  Values: [0.03594432454327817, 0.46373792229178706]
  Params: {'input_force': 2.0432698455818414}


In [ ]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps,
    0.01,
    matrix_size,
    M_INV,
    DAMP,
    U * study.best_trials[0].params["input_force"],
    nodes_pos,
    connections_list,
    k_vals,
    [0, 1, 3, 4],
    100,
)

Y_test, Y_pred = hyper_param_input(study.best_trials[0].params["input_force"])
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
r_2, mse

(0.03594432454327817, 0.46373792229178706)

In [ ]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
scatter_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
).show()

# Modes for Full Constraint

What I see is that the movement for a specific K just rotates it kinda. Since theorticlly for that speicfic K it's not possible to push it into the inside of the hexagon. Since the specific K just brings it back to that specific spot no matter what.

In [825]:
node_m = np.ones(num_nodes)
m_diag = np.repeat(node_m, dims)
M = np.diag(m_diag)

k_vals = np.ones(src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))
K = create_stiffness_matrix(nodes_pos, connections_list, k_vals)


constrained_val = 1
for node in [0, 1, 2, 3, 4, 5]:
    idx = node * dims
    K[idx, idx] += constrained_val
    K[idx + 1, idx + 1] += constrained_val

In [826]:
eigenvalues, eigenvectors = eigh(K, M)
eigenvalues = np.abs(eigenvalues)
eigenvalues

array([1. , 1. , 1. , 1. , 1. , 1. , 2. , 2.5, 2.5, 3.5, 3.5, 4. ])

In [827]:
fig = fpl.Figure(canvas="glfw")

node_pos_modes = []
dots = []
col = 4
seperation = 3
color_list = ["cyan", "magenta", "yellow", "white", "red", "green", "blue", "orange"]
for i in range(len(eigenvalues)):
    node_pos_2D = nodes_pos.astype(np.float32)
    node_pos_2D[:, 0] += seperation * (i % col)
    node_pos_2D[:, 1] += -seperation * (i // col)
    node_pos_modes.append(node_pos_2D)
    hex_color = color_list[i % len(color_list)]
    dots.append(fig[0, 0].add_scatter(data=node_pos_2D, sizes=10, colors=hex_color))

step = 0
dt = 0.01
max_mov = 0.25


def update_springs():
    global step
    step += 1
    for i in range(len(eigenvalues)):
        mode_vec = eigenvectors[:, i]
        max_val = np.max(np.abs(mode_vec))
        if max_val < 1e-6:
            max_val = 1.0
        scaled_mode = (mode_vec / max_val) * max_mov
        disp = scaled_mode * np.sin(np.sqrt(eigenvalues[i]) * step * dt)
        disp_2D = disp.reshape(-1, 2).astype(np.float32)
        coords = node_pos_modes[i] + disp_2D
        dots[i].data[:, :2] = coords.astype(np.float32)


fig.add_animations(update_springs)
fig.show()

In [ ]:
fig = fpl.Figure(canvas="glfw")

node_pos_modes = []
dots = []
col = 4
seperation = 3
color_list = ["cyan", "magenta", "yellow", "white", "red", "green", "blue", "orange"]
for i in range(len(eigenvalues)):
    node_pos_2D = nodes_pos.astype(np.float32)
    node_pos_2D[:, 0] += seperation * (i % col)
    node_pos_2D[:, 1] += -seperation * (i // col)
    node_pos_modes.append(node_pos_2D)
    hex_color = color_list[i % len(color_list)]
    dots.append(fig[0, 0].add_scatter(data=node_pos_2D, sizes=10, colors=hex_color))

step = 0
dt = 0.01
max_mov = 0.25


def update_springs():
    global step
    step += 1
    for i in range(len(eigenvalues)):
        mode_vec = eigenvectors[:, i]
        max_val = np.max(np.abs(mode_vec))
        if max_val < 1e-6:
            max_val = 1.0
        scaled_mode = (mode_vec / max_val) * max_mov
        disp = scaled_mode * np.sin(np.sqrt(eigenvalues[i]) * step * dt)
        disp_2D = disp.reshape(-1, 2).astype(np.float32)
        coords = node_pos_modes[i] + disp_2D
        dots[i].data[:, :2] = coords.astype(np.float32)


fig.add_animations(update_springs)
fig.show()

# Force Curve?

I was wondering if it's possible to show the displacement graph. Since the issue now is that K(x) is a function of x and is non linear right. I want to show that for a big enough force it swaps into a new displacement state kinda. Like it moves on right of the hexagon then enough force it gets pushed to the inside.

I don't know how exactly I can get the exact displacment. Lowkey just got an idea each x axis is the force. Then I plot the whole displacement. Then I should see the max and min which should be the important bit.

Maybe do velocity how it kinda increase for a force that forces it to change. With no damping. Or just have no damping for both tbh. But once it swaps. Nah without damping it would keep swaping tbh.

Maybe I can plot each point and it's force for each K stiff then we have a vector plot of force depedning on pos.

In [ ]:
side_len = 1
x = np.array(
    [-side_len / 2, side_len / 2, side_len, side_len / 2, -side_len / 2, -side_len]
)
y = np.array(
    [
        0,
        0,
        side_len * np.sqrt(3) / 2,
        side_len * np.sqrt(3),
        side_len * np.sqrt(3),
        side_len * np.sqrt(3) / 2,
    ]
)
nodes_pos = np.column_stack((x, y))

In [ ]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = np.ones(num_nodes) * .1
m_diag = np.repeat(node_m, dims)
m_inv_diag = 1.0 / m_diag
M = np.diag(m_diag)
M_INV = np.diag(m_inv_diag)

node_c = np.ones(num_nodes) * .1
c_diag = np.repeat(node_c, dims)
DAMP = np.diag(c_diag)

In [ ]:
# We are saying for the target node or the right node
# we apply a force of 1 in first step in x dir
force = 1000
U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = np.array([2])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
vectorized_force[0, 0] = force
U[:, col_indices] = vectorized_force

In [ ]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, 1)

rng = np.random.default_rng(42)
k_vals = np.ones(src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [ ]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps,
    0.001,
    matrix_size,
    M_INV,
    DAMP,
    U,
    nodes_pos,
    connections_list,
    k_vals,
    constrained_nodes=[0, 1, 3, 4],
    wall_nodes=[-1],
    constrained_values=100,
)

X = np.column_stack((displacement, velocity))

In [ ]:
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=10000,
    vel=velocity,
    steps_jump=3
).show()